# ASR comparison

Runs the synthetic clip set (Chinese, English, Japanese, and two mixed-language clips) through each model and compares accuracy and speed.

- Results are stored under `results/asr/<chip>/`. Running a model only executes cases that are new or stale, so rerunning this notebook is cheap.
- `x_realtime` is audio seconds divided by inference seconds; higher is faster. `error` is WER for English and whitespace-insensitive CER for everything else; lower is better.
- Models stay loaded in memory for the duration of a run, so timings exclude loading. The first run of a model downloads its weights from Hugging Face.

In [ ]:
from dito_bench.clips import ensure_clips

for clip in ensure_clips():
    print(f"{clip.name:<16} [{clip.lang:<5}] {clip.text}")

In [ ]:
import pandas as pd

from dito_bench.asr import benchmark
from dito_bench.asr.registry import default_asr_keys

pd.set_option("display.max_colwidth", 48)
keys = default_asr_keys()
benchmark.run(keys)
rows, warnings = benchmark.report_rows(keys)
print("\n".join(warnings))
results = pd.DataFrame(rows)
results

In [ ]:
pd.DataFrame(benchmark.summarize(rows))

## Hotwords

Qwen3-ASR biases decoding with a hotword list passed as context. Whisper only supports a soft bias through `initial_prompt`, and SenseVoice has no such mechanism. The clip `zh_hotwords_01` is built from rare terms that are easy to confuse with homophones; the `+hotwords` case runs the same audio with the terms supplied.

In [ ]:
results[results["clip"].str.startswith("zh_hotwords_01")][["model", "clip", "error", "text"]]

## Your own recording

Defaults to the newest file in `recordings/`, or a synthetic clip when there is none. Replace `audio` with any 16 kHz WAV path. Nothing is stored for this comparison because there is no reference text.

In [ ]:
from dito_bench.paths import AUDIO_DIR
from dito_bench.recorder import latest_recording

audio = latest_recording() or AUDIO_DIR / "ja_en_mix_01.wav"
print("audio:", audio)
pd.DataFrame(benchmark.compare_file(audio, keys))